# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

# Load & Inpect data

In [0]:
df_raw = spark.table("workspace.bronze.erp_px_cat_g1v2_raw")
df_raw.display()

In [0]:
print(f'There are {df_raw.count()} rows')
print(f'Table contains {df_raw.select("ID").distinct().count()} distinct ids')
df_raw.printSchema()

# Transform data

## Rename columns

In [0]:
name_map = {
    'ID': 'category_key',
    'CAT': 'category_name',
    'SUBCAT': 'subcategory_name',
    'MAINTENANCE': 'maintenance'
}

df = df_raw

for old, new in name_map.items():
    df = df.withColumnRenamed(old, new)
    
df.printSchema()

## Align 'category_key'

In [0]:
df = df.withColumn('category_key', F.regexp_replace('category_key', '_', '-'))

df.display()
df.select('category_key').distinct().display()
df.select('subcategory_name').distinct().display()

## Trim string columns

In [0]:
df.schema

In [0]:
df =  df.select([
    F.trim(F.col(f.name)).alias(f.name) if isinstance(f.dataType, StringType)
    else F.col(f.name)
    for f in df.schema.fields
])

## Cast datatype for 'maintaince'

In [0]:
df = df.withColumn(
  'maintenance',
  F.when(F.lower(F.col('maintenance')) == 'yes', F.lit(True))
  .when(F.lower(F.col('maintenance')) == 'no', F.lit(False))
  .otherwise(F.col('maintenance'))
)
df.display()

# Write table

In [0]:
df.write.mode('overwrite').saveAsTable('workspace.silver.erp_category')

## Sanity check

In [0]:
%sql
SELECT * FROM workspace.silver.erp_category